# 🤖 Complete RAG System — Practical Learning Notebook

**Book used:** *As a Man Thinketh* by James Allen (1903) — 100% Free & Public Domain  
**LLM:** Ollama running locally (no API key needed, no internet required)  
**Framework:** LangChain  
**Vector Store:** ChromaDB  

---

## 📋 What You Will Build Step by Step:
```
📄 PDF/Text Document
       ↓
✂️  Chunking (split into pieces)
       ↓
🔢 Embeddings (convert text → numbers)
       ↓
🗄️  Vector Store (ChromaDB — store the numbers)
       ↓
🔍 Retrieval (find relevant chunks for a question)
       ↓
🦙 LLM via Ollama (generate answer using retrieved context)
       ↓
💬 Chat Interface (terminal-based Q&A loop)
```

---
⚠️ **Pre-requisite:** Make sure Ollama is installed and running. See `setup/OLLAMA_SETUP.md`


---
## 🔧 CELL 1 — Install All Dependencies
Run this ONCE. After installation, you don't need to run it again.

In [ ]:
# Install all required packages
import subprocess, sys

packages = [
    "langchain>=0.2.0",
    "langchain-community>=0.2.0",
    "langchain-ollama>=0.1.0",
    "chromadb>=0.5.0",
    "pypdf>=4.0.0",
    "requests",
    "tqdm",
]

for pkg in packages:
    print(f"📦 Installing {pkg}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("\n✅ All packages installed successfully!")

---
## 📥 CELL 2 — Download the Book (Public Domain)
We download *As a Man Thinketh* by James Allen from Project Gutenberg.  
This book is 100% free — published in 1903, it is in the public domain worldwide.

**Why this book?**  
- Free to use legally  
- Short enough to process quickly  
- Great content about mindset, discipline, and success — similar themes to self-help books you enjoy

In [ ]:
import requests
import os

# ─────────────────────────────────────────────────────────────
# Download the public domain book from Project Gutenberg
# ─────────────────────────────────────────────────────────────
BOOK_URL = "https://www.gutenberg.org/cache/epub/4507/pg4507.txt"
DATA_DIR = "../data"
BOOK_PATH = os.path.join(DATA_DIR, "as_a_man_thinketh.txt")

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(BOOK_PATH):
    print("📥 Downloading book from Project Gutenberg...")
    response = requests.get(BOOK_URL)
    response.raise_for_status()
    with open(BOOK_PATH, "w", encoding="utf-8") as f:
        f.write(response.text)
    print(f"✅ Book downloaded and saved to: {BOOK_PATH}")
else:
    print(f"✅ Book already exists at: {BOOK_PATH}")

# Show file size and preview
size_kb = os.path.getsize(BOOK_PATH) / 1024
print(f"📄 File size: {size_kb:.1f} KB")

with open(BOOK_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"📝 Total characters: {len(raw_text):,}")
print("\n--- BOOK PREVIEW (first 500 characters) ---")
print(raw_text[:500])

---
## 🔍 CELL 3 — Understanding: What is a Document Loader?

In RAG, the first step is **loading** your document. LangChain provides many loaders:
- `TextLoader` — for `.txt` files  
- `PyPDFLoader` — for `.pdf` files  
- `WebBaseLoader` — for websites  
- `CSVLoader` — for CSV files  

A **Document** in LangChain has two parts:
- `page_content` — the actual text
- `metadata` — info about the document (source, page number, etc.)

In [ ]:
from langchain_community.document_loaders import TextLoader

# ─────────────────────────────────────────────────────────────
# STEP 1: Load the document
# ─────────────────────────────────────────────────────────────
print("📂 Loading document...")
loader = TextLoader(BOOK_PATH, encoding="utf-8")
documents = loader.load()

print(f"✅ Loaded {len(documents)} document(s)")
print(f"📝 Total characters in document: {len(documents[0].page_content):,}")
print(f"🏷️  Metadata: {documents[0].metadata}")
print("\n--- First 300 characters of content ---")
print(documents[0].page_content[:300])

---
## ✂️ CELL 4 — CHUNKING: Splitting Text into Pieces

### Why do we chunk?
LLMs have a **context window limit** — they can only process a limited amount of text at once.  
Also, embedding a huge document as one unit loses detail.  
By chunking, we split the book into small, meaningful pieces so we can:
1. Embed each piece accurately
2. Retrieve only the **relevant** piece for a given question

### Key Parameters:
| Parameter | What it controls |
|-----------|------------------|
| `chunk_size` | Maximum characters per chunk |
| `chunk_overlap` | How many characters overlap between chunks (prevents cutting off context) |

### Visualizing chunks:
```
Full Text:  [=========================== entire book ===========================]
Chunk 1:    [======]          (chunk_size = 6 chars, for example)
Chunk 2:       [======]       (overlaps with chunk 1 by chunk_overlap chars)
Chunk 3:          [======]
```

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ─────────────────────────────────────────────────────────────
# STEP 2: Chunk the document
# ─────────────────────────────────────────────────────────────

# RecursiveCharacterTextSplitter is the recommended splitter.
# It tries to split on: paragraphs → sentences → words → characters
# This preserves natural language boundaries as much as possible.

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,        # Each chunk has at most 800 characters
    chunk_overlap=150,     # 150 characters shared between adjacent chunks
    length_function=len,   # How to measure "length" (character count here)
    separators=["\n\n", "\n", " ", ""]  # Try these separators in order
)

chunks = text_splitter.split_documents(documents)

print(f"📊 Chunking Results:")
print(f"   Original document: 1 document, {len(documents[0].page_content):,} characters")
print(f"   After chunking:    {len(chunks)} chunks")
print(f"   Average chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} characters")
print()

# Show a few example chunks to understand what they look like
for i, chunk in enumerate(chunks[5:8], start=5):
    print(f"--- CHUNK #{i} ({len(chunk.page_content)} chars) ---")
    print(chunk.page_content)
    print()

---
## 🔢 CELL 5 — EMBEDDINGS: Converting Text to Numbers

### What are Embeddings?
An **embedding** is a list of numbers (a vector) that represents the *meaning* of a piece of text.  
Text with similar meaning → vectors that are mathematically close to each other.

```
"I wake up at 5am"  →  [0.12, -0.34, 0.89, 0.01, ...]  (384 numbers)
"Early morning rise" →  [0.11, -0.30, 0.85, 0.03, ...]  (similar!)
"The cat sat"       →  [-0.88, 0.45, -0.12, 0.67, ...] (very different)
```

### Why use Ollama for Embeddings?
- **Free** — no API costs
- **Private** — your data never leaves your computer
- **Fast** — runs locally

We use `nomic-embed-text` — a powerful embedding model that runs locally via Ollama.

> ⚠️ **First run this in your terminal:** `ollama pull nomic-embed-text`

In [ ]:
from langchain_ollama import OllamaEmbeddings

# ─────────────────────────────────────────────────────────────
# STEP 3: Create the embedding model
# ─────────────────────────────────────────────────────────────

# This uses Ollama running locally on port 11434
# Make sure: ollama pull nomic-embed-text   (done in setup)
embedding_model = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url="http://localhost:11434"  # Default Ollama URL
)

# Test: embed a single sentence and inspect the vector
test_sentence = "A man's mind is like a garden that must be tended."
print(f"📝 Test sentence: '{test_sentence}'")
print("🔄 Generating embedding...")

vector = embedding_model.embed_query(test_sentence)

print(f"\n✅ Embedding generated!")
print(f"📏 Vector dimensions: {len(vector)}")
print(f"🔢 First 10 values: {[round(v, 4) for v in vector[:10]]}")
print()
print("💡 INSIGHT: Each text becomes a list of numbers.")
print("   Similar texts → similar numbers → can be found by similarity search!")

---
## 🗄️ CELL 6 — VECTOR STORE: Storing & Searching Embeddings

### What is a Vector Store?
A **vector store** (also called vector database) stores your embeddings and allows **similarity search**.  
When you ask a question:
1. Your question → embedded into a vector
2. Vector store searches for the **closest** chunk vectors
3. Returns the most relevant text chunks

### We use ChromaDB:
- Runs completely in-memory or on disk
- No external server needed
- Perfect for learning and prototyping

```
Vector Store (ChromaDB):
┌─────────────────────────────────────────────┐
│  chunk_id_1  │  "Man is made or unmade..."  │  [0.12, -0.34, ...]  │
│  chunk_id_2  │  "The body is servant..."    │  [0.88, 0.21, ...]   │
│  chunk_id_3  │  "Thought and character..."  │  [0.05, -0.67, ...]  │
│     ...      │         ...                  │        ...           │
└─────────────────────────────────────────────┘
```

In [ ]:
from langchain_community.vectorstores import Chroma
import os

# ─────────────────────────────────────────────────────────────
# STEP 4: Create the Vector Store
# ─────────────────────────────────────────────────────────────

CHROMA_DIR = "../data/chroma_db"  # Where ChromaDB will persist on disk

print("🔄 Creating vector store...")
print(f"   📦 Embedding {len(chunks)} chunks using nomic-embed-text...")
print("   ⏳ This may take 1-3 minutes on first run (embedding each chunk)...")
print()

# This call:
# 1. Takes each chunk
# 2. Sends it to Ollama to get an embedding vector
# 3. Stores the text + vector in ChromaDB
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=CHROMA_DIR,
    collection_name="book_knowledge"
)

print(f"✅ Vector store created!")
print(f"📂 Persisted to disk at: {CHROMA_DIR}")
print(f"📊 Total chunks stored: {vectorstore._collection.count()}")

---
## 🔍 CELL 7 — RETRIEVAL: Finding Relevant Chunks

### How Retrieval Works:
```
User Question: "How does thought affect our circumstances?"
       ↓
Embed the question: [0.23, -0.41, 0.77, ...]
       ↓
Search ChromaDB for closest vectors
       ↓
Return top-k most similar chunks (k=3 or 4)
```

### Similarity Score:
We use **cosine similarity** — measures the angle between two vectors.
- Score = 1.0 → identical meaning
- Score = 0.0 → no relation

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 5: Test Retrieval
# ─────────────────────────────────────────────────────────────

# Create a retriever from the vector store
# k=3 means: return the 3 most relevant chunks
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# Test with a sample question
test_question = "What is the relationship between thought and character?"
print(f"❓ Test Question: {test_question}")
print()

retrieved_docs = retriever.invoke(test_question)

print(f"📚 Retrieved {len(retrieved_docs)} relevant chunks:")
print("=" * 60)
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n[CHUNK {i}]")
    print(doc.page_content)
    print(f"Source: {doc.metadata.get('source', 'unknown')}")
    print("-" * 40)

print()
print("💡 These chunks will be given to the LLM as CONTEXT to answer your question!")

---
## 🦙 CELL 8 — LLM Setup: Loading Ollama

### What is Ollama?
Ollama lets you run powerful LLMs (like Llama 3, Mistral, Gemma) completely on your local machine:
- **No API key** needed
- **No internet** needed after download
- **Free forever**
- **Private** — your data stays on your machine

### Model we use: `llama3.2`
- Small enough to run on most laptops (4-8GB RAM)
- Smart enough for Q&A tasks
- Good English understanding

> ⚠️ **First run this in your terminal:** `ollama pull llama3.2`

In [ ]:
from langchain_ollama import OllamaLLM

# ─────────────────────────────────────────────────────────────
# STEP 6: Initialize the LLM
# ─────────────────────────────────────────────────────────────

llm = OllamaLLM(
    model="llama3.2",           # The LLM model to use
    base_url="http://localhost:11434",
    temperature=0.1,            # Low = more factual, High = more creative
    # temperature=0 → deterministic answers (same question → same answer)
    # temperature=1 → creative/random answers
)

# Quick test to verify Ollama is working
print("🔄 Testing Ollama connection...")
test_response = llm.invoke("Say 'Ollama is working!' and nothing else.")
print(f"✅ Ollama Response: {test_response}")
print()
print("🦙 LLM is ready! Model: llama3.2")

---
## 🔗 CELL 9 — RAG CHAIN: Connecting Everything Together

### The Full RAG Pipeline:
```
Question → Retriever → [Context Chunks] → Prompt Template → LLM → Answer
```

### The Prompt Template:
The **prompt template** is how we tell the LLM:
1. Here is the context from the book
2. Here is the user's question
3. Answer based on the context

This is what **prevents hallucination** — the LLM is grounded in real book content!

In [ ]:
from langchain.prompts import PromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

# ─────────────────────────────────────────────────────────────
# STEP 7: Build the RAG Chain
# ─────────────────────────────────────────────────────────────

# The prompt template tells the LLM HOW to use the context
RAG_PROMPT_TEMPLATE = """
You are a helpful assistant that answers questions based on the provided context from the book "As a Man Thinketh" by James Allen.

Use ONLY the information from the context below to answer the question.
If the answer is not in the context, say: "I don't have enough information from the book to answer that."
Keep your answers clear, concise, and insightful.

CONTEXT FROM THE BOOK:
{context}

QUESTION: {question}

ANSWER:"""

prompt = PromptTemplate(
    template=RAG_PROMPT_TEMPLATE,
    input_variables=["context", "question"]
)

# Helper function to format retrieved docs into a single string
def format_docs(docs):
    return "\n\n---\n\n".join([doc.page_content for doc in docs])

# Build the RAG chain using LangChain Expression Language (LCEL)
# This is a pipeline: each | passes output to the next step
rag_chain = (
    {
        "context": retriever | format_docs,  # Retrieve + format context
        "question": RunnablePassthrough()    # Pass question through unchanged
    }
    | prompt          # Fill the prompt template
    | llm             # Send to Ollama LLM
    | StrOutputParser() # Parse output as string
)

print("✅ RAG Chain built successfully!")
print()
print("Pipeline summary:")
print("  1. User question → Retriever (finds 3 relevant chunks)")
print("  2. Chunks → format_docs (joins them into one string)")
print("  3. context + question → PromptTemplate (fills the template)")
print("  4. Filled prompt → OllamaLLM (generates answer)")
print("  5. LLM output → StrOutputParser (returns clean string)")

---
## 💬 CELL 10 — Test the RAG System with Sample Questions

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 8: Test with sample questions
# ─────────────────────────────────────────────────────────────

test_questions = [
    "What does James Allen say about the relationship between thought and character?",
    "How does the mind affect our health according to the book?",
    "What is the significance of having a definite purpose in life?",
]

for i, question in enumerate(test_questions, 1):
    print(f"{'='*60}")
    print(f"❓ QUESTION {i}: {question}")
    print(f"{'='*60}")
    print("🔄 Thinking...")
    
    answer = rag_chain.invoke(question)
    
    print(f"\n🤖 ANSWER:")
    print(answer)
    print()

---
## 💬 CELL 11 — Interactive Chat Loop

This is the final product! Run this cell to start chatting with the book.  
Type **'quit'** or **'exit'** to stop.

### How it works:
1. You type a question
2. The system finds the 3 most relevant book passages
3. Passes them + your question to the LLM
4. LLM generates an answer grounded in the book

### With Memory (Conversation History):
We also add **conversation memory** so the LLM can reference previous answers!

In [ ]:
from langchain.memory import ConversationBufferWindowMemory
from langchain.prompts import PromptTemplate
from langchain_ollama import OllamaLLM

# ─────────────────────────────────────────────────────────────
# STEP 9: RAG with Conversation Memory
# ─────────────────────────────────────────────────────────────

# Memory stores the last 5 exchanges
memory = ConversationBufferWindowMemory(
    k=5,
    memory_key="chat_history",
    return_messages=False
)

# Updated prompt with chat history
CHAT_PROMPT_TEMPLATE = """
You are a wise assistant helping someone understand the teachings from "As a Man Thinketh" by James Allen.
Answer questions using ONLY the provided book context.
If the answer is not in the context, say so honestly.
Be thoughtful, clear, and inspiring in your responses.

PREVIOUS CONVERSATION:
{chat_history}

RELEVANT BOOK CONTEXT:
{context}

CURRENT QUESTION: {question}

ANSWER:"""

chat_prompt = PromptTemplate(
    template=CHAT_PROMPT_TEMPLATE,
    input_variables=["chat_history", "context", "question"]
)

# ─────────────────────────────────────────────────────────────
# INTERACTIVE CHAT LOOP
# ─────────────────────────────────────────────────────────────

print("╔══════════════════════════════════════════════════════════════╗")
print("║          📖 RAG CHATBOT — As a Man Thinketh                  ║")
print("║          Powered by Ollama (llama3.2) + LangChain            ║")
print("╠══════════════════════════════════════════════════════════════╣")
print("║  Ask anything about the book's teachings!                    ║")
print("║  Type 'quit' or 'exit' to stop.                              ║")
print("║  Type 'sources' after a question to see retrieved chunks.    ║")
print("╚══════════════════════════════════════════════════════════════╝")
print()

show_sources = False
last_sources = []

while True:
    # Get user input
    user_input = input("\n🧑 You: ").strip()
    
    # Check for exit commands
    if user_input.lower() in ["quit", "exit", "q"]:
        print("\n👋 Goodbye! Keep thinking great thoughts!")
        break
    
    # Show sources from last query
    if user_input.lower() == "sources":
        if last_sources:
            print("\n📚 Sources used in last answer:")
            for i, src in enumerate(last_sources, 1):
                print(f"\n[Source {i}]")
                print(src.page_content[:300] + "...")
        else:
            print("No previous sources to show.")
        continue
    
    if not user_input:
        continue
    
    # Retrieve relevant chunks
    relevant_docs = retriever.invoke(user_input)
    last_sources = relevant_docs
    context = format_docs(relevant_docs)
    
    # Get conversation history
    chat_history = memory.load_memory_variables({})["chat_history"]
    
    # Build the prompt
    filled_prompt = chat_prompt.format(
        chat_history=chat_history,
        context=context,
        question=user_input
    )
    
    # Get LLM response
    print("\n🤖 Assistant: ", end="", flush=True)
    response = llm.invoke(filled_prompt)
    print(response)
    
    # Save to memory
    memory.save_context(
        {"input": user_input},
        {"output": response}
    )
    
    print("\n💡 Tip: Type 'sources' to see which book passages were used.")

---
## 🔬 CELL 12 — BONUS: Load from Disk (Skip Re-embedding)

Next time you open this notebook, you don't need to re-embed everything!  
Just load the existing ChromaDB from disk:

In [ ]:
# ─────────────────────────────────────────────────────────────
# BONUS: Load existing vector store from disk
# Run this instead of CELL 6 on subsequent runs
# ─────────────────────────────────────────────────────────────

from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings

CHROMA_DIR = "../data/chroma_db"

embedding_model = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url="http://localhost:11434"
)

# Load from disk (no re-embedding needed!)
vectorstore = Chroma(
    persist_directory=CHROMA_DIR,
    embedding_function=embedding_model,
    collection_name="book_knowledge"
)

print(f"✅ Loaded vector store from disk!")
print(f"📊 Chunks available: {vectorstore._collection.count()}")

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("🔍 Retriever ready!")

---
## 🧠 CELL 13 — BONUS: Visualize Chunk Similarity

See similarity scores to understand what the retriever is doing internally.

In [ ]:
# ─────────────────────────────────────────────────────────────
# BONUS: See similarity scores for retrieved docs
# ─────────────────────────────────────────────────────────────

question = "How can a person change their life circumstances?"

# similarity_search_with_relevance_scores returns (doc, score) pairs
docs_with_scores = vectorstore.similarity_search_with_relevance_scores(
    question, 
    k=5
)

print(f"❓ Question: {question}")
print(f"\n📊 Top 5 results with similarity scores:")
print("=" * 70)

for i, (doc, score) in enumerate(docs_with_scores, 1):
    bar_length = int(score * 30)
    bar = "█" * bar_length + "░" * (30 - bar_length)
    print(f"\n#{i} | Score: {score:.4f} |{bar}|")
    print(f"    {doc.page_content[:200]}...")

print()
print("💡 Higher score = more relevant to your question")
print("   Scores range from 0.0 (unrelated) to 1.0 (identical meaning)")

---
## 🎉 You've Built a Complete RAG System!

### What you accomplished:
| Step | Technology | What it does |
|------|-----------|---------------|
| 1. Load | LangChain TextLoader | Reads the book into Documents |
| 2. Chunk | RecursiveCharacterTextSplitter | Splits into 800-char pieces with 150-char overlap |
| 3. Embed | Ollama nomic-embed-text | Converts each chunk to a 768-dim vector |
| 4. Store | ChromaDB | Persists vectors on disk for fast search |
| 5. Retrieve | Similarity Search | Finds the 3 most relevant chunks per question |
| 6. Generate | Ollama llama3.2 | Answers using retrieved context |
| 7. Chat | Memory + Loop | Multi-turn conversation with history |

### Next Steps to Explore:
- Try different `chunk_size` values (500, 1000, 1500) and see how answers change
- Try `k=5` in the retriever for more context
- Try a different LLM: `ollama pull mistral` and change the model name
- Load a PDF instead of text using `PyPDFLoader`
- Try `HuggingFaceEmbeddings` for alternative embeddings

📖 Read the full guide: `guides/RAG_Complete_Beginner_Guide.md`